Build a Simple LLM Application with LCEL

In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!
After seeing this, you'll have a high level overview of:
* Using language models
* Using PromptTemplates and OutputParsers
* Using Lang Chain Expression Language (LCEL) to chain components together
* Debugging and tracing your application using LangSmith
* Deploying your application with LangServe

In [2]:
import os
from dotenv import load_dotenv
load_dotenv

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

groq_api_key = os.getenv("GROQ_API_KEY")
print(groq_api_key)

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


In [10]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant", groq_api_key=groq_api_key)
llm

# llm = ChatGroq(model="llama-3.1-8b-instant")

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000029DFDC48A50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000029DFDC49450>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [8]:
result = llm.invoke("Hi How are you?")
result

AIMessage(content="I'm doing well, thank you for asking. I'm a large language model, so I don't have emotions or feelings like humans do, but I'm here to help answer any questions or provide information you might need. How can I assist you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 40, 'total_tokens': 93, 'completion_time': 0.055745575, 'completion_tokens_details': None, 'prompt_time': 0.001862384, 'prompt_tokens_details': None, 'queue_time': 0.046338136, 'total_time': 0.057607959}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0a53-6338-7f73-adff-2eb78e8e770c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 40, 'output_tokens': 53, 'total_tokens': 93})

In [21]:
from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()
output_parser.invoke(result)

"I'm doing well, thank you for asking. I'm a large language model, so I don't have emotions or feelings like humans do, but I'm here to help answer any questions or provide information you might need. How can I assist you today?"

In [19]:
### Chains --> prompt | llm | StroutputParser

from langchain_core.messages import HumanMessage,SystemMessage,AIMessage

messages = [
    SystemMessage(content="Translate the following from English to Spanish"),
    HumanMessage(content="Hello how are you?")
]

ai_response = llm.invoke(messages)
ai_response

AIMessage(content='Hola, ¿cómo estás?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 47, 'total_tokens': 56, 'completion_time': 0.015723535, 'completion_tokens_details': None, 'prompt_time': 0.002261488, 'prompt_tokens_details': None, 'queue_time': 0.046792653, 'total_time': 0.017985023}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0a5a-fcf1-71b3-acbf-7b33a4de0153-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 47, 'output_tokens': 9, 'total_tokens': 56})

In [20]:
output_parser.invoke(ai_response)

'Hola, ¿cómo estás?'

In [ ]:
# Using LCEL - Chain the components - Instead of two steps we can make use of chain in langchain

chain = llm | output_parser
chain.invoke(messages)

'Hola, ¿cómo estás?'

In [32]:
### prompt Templates

from langchain_core.prompts import ChatPromptTemplate

generic_template = "Translate the following into {language}"

prompt = ChatPromptTemplate.from_messages([
                ("system", generic_template),
                ("user", "{text}")
            ]
)

result

ChatPromptValue(messages=[SystemMessage(content='Transalate the following into french', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})])

In [34]:
result = prompt.invoke({"language":"French","text":"Hello"})
result.to_messages()

[SystemMessage(content='Translate the following into French', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [35]:
chain_new = prompt | llm | output_parser

chain_new.invoke({"language":"French","text":"Hello"})

'Bonjour.'

### LangServe

- LangServe helps developers deploy Langchain runnables and chains as a REST API
- Library is integrated with FastAPI and uses pydantic for data validation

In [1]:
### Check in serve.py